In [9]:
import requests
import os
import glob
import pandas as pd
from os.path import join

In [10]:
def download_nasa_data(cities_coords):
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    os.makedirs("nasa_data", exist_ok=True)

    for municipality in cities_coords.itertuples():
        params = {
            "parameters": "PRECTOTCORR,T2M,T2MDEW,RH2M,TOA_SW_DWN,ALLSKY_SFC_SW_DWN",
            "community": "AG",                    
            "longitude": municipality.longitude,
            "latitude": municipality.latitude,
            "start": "20030101",                  
            "end": "20241231",
            "format": "CSV",                       
            'header': 'true'                      
        }

        response = requests.get(base_url, params=params)

        if response.status_code == 200:
            citie_name = str(municipality.name).replace(" ", "_").replace("/", "-")
            filename = f"nasa_climate_data_{municipality.codigo_ibge}-{municipality.name}.csv"
            with open(join("nasa_data", filename), "wb") as f:
                f.write(response.content)
            print(f"Download successful: {filename}")
        else:
            print(f"Error for {municipality.name} (code: {municipality.code}): {response.status_code} {response.text}")

In [11]:
#download_nasa_data('ibge/cities_to_analize.csv')

In [20]:
def read_nasa_data(input_folder):
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

    all_city_data = []
    for csv_path in csv_files:
        filename = os.path.basename(csv_path)
        city_name = filename.replace('nasa_climate_data_', '').replace('.csv', '').split('-')[1].strip()

        df_raw = pd.read_csv(csv_path, sep=',', skiprows=14)
        df_raw['city'] = city_name
        df_raw['code'] = int(filename.split('-')[0].split('_')[-1])

        df_raw['DATE'] = pd.to_datetime(df_raw['YEAR'].astype(str) + df_raw['DOY'].astype(str), format='%Y%j').dt.strftime('%Y-%m-%d')

        df_raw.drop(columns=['DOY'], inplace=True)

        all_city_data.append(df_raw)
        
    df_raw = pd.concat(all_city_data, ignore_index=True)
    df_raw['DATE'] = pd.to_datetime(df_raw['DATE'])
    df_raw.set_index('DATE', inplace=True)
        
    return df_raw

In [13]:
def process_nasa_data(df_raw):

    df_filtered = df_raw[(df_raw.index.month >= 1) & (df_raw.index.month <= 7)].copy()

    df_filtered['global_radiation_kj_m2'] = df_filtered['ALLSKY_SFC_SW_DWN'] * 3600

    df_filtered['year'] = df_filtered.index.year

    yearly_summary = df_filtered.groupby(['year', 'city', 'code']).agg(
        mean_temperature_c=('T2M', 'mean'),  
        max_temperature_c=('T2M', 'max'),    
        min_temperature_c=('T2M', 'min'),    
        total_rain_mm=('PRECTOTCORR', 'sum'),
        sum_global_radiation_kj_m2=('ALLSKY_SFC_SW_DWN', 'sum'),
        mean_relative_humidity_pct=('RH2M', 'mean'),
        max_relative_humidity_pct=('RH2M', 'max'),
        min_relative_humidity_pct=('RH2M', 'min')
    ).reset_index().round(2)
    
    return yearly_summary

In [21]:
df_nasa_raw = read_nasa_data("nasa")

In [22]:
len(df_nasa_raw)

345548

In [23]:
df_nasa_final = process_nasa_data(df_nasa_raw)

In [24]:
df_nasa_final

,year,city,code,mean_temperature_c,max_temperature_c,min_temperature_c,total_rain_mm,sum_global_radiation_kj_m2,mean_relative_humidity_pct,max_relative_humidity_pct,min_relative_humidity_pct
0,2003,alto araguaia,5100300,21.36,25.64,13.69,1005.70,3811.66,79.53,92.51,53.16
1,2003,alto garcas,5100409,21.67,25.71,14.36,1004.79,3874.67,80.56,93.61,58.87
2,2003,alto taquari,5100607,20.80,25.22,13.06,1035.33,3811.66,81.27,93.96,58.80
3,2003,brasnorte,5101902,23.88,26.56,20.78,1126.40,3830.94,87.58,96.82,55.13
4,2003,campo novo do parecis,5102637,22.86,26.01,17.33,1060.32,3858.95,83.84,93.95,58.19
...,...,...,...,...,...,...,...,...,...,...,...
941,2024,tangara da serra,5107958,26.69,29.89,19.37,683.09,4055.10,66.16,91.41,29.61
942,2024,tapurah,5108006,25.35,27.79,21.72,905.62,4133.81,79.76,97.98,47.78
943,2024,tesouro,5108105,26.73,31.19,22.68,376.77,4079.75,55.86,90.58,25.44
944,2024,uniao do sul,5108303,25.83,27.91,23.10,850.41,4140.89,79.65,97.85,47.55


In [26]:
%store df_nasa_final

Stored 'df_nasa_final' (DataFrame)


In [25]:
len(df_nasa_final)

946